In [5]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Flatten, Dense, Conv2D, MaxPooling2D, BatchNormalization, ReLU
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Functional api 모델 정의

In [7]:
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x
    
inputs = Input(shape=(32, 32, 3))

# stage1
x = conv_block(inputs, 32)
x = conv_block(x, 32)
x = MaxPooling2D()(x)

# stage2
x = conv_block(x, 64)
x = conv_block(x, 64)
x = MaxPooling2D()(x)

# stage3
x = conv_block(x, 128)
x = conv_block(x, 128)
x = MaxPooling2D()(x)

# 분류기
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(10, activation='softmax')(x)

model = Model(inputs, outputs, name='CIFAR10_CNN')
print(model.summary())

Model: "CIFAR10_CNN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 conv2d_6 (Conv2D)           (None, 32, 32, 32)        864       
                                                                 
 batch_normalization_6 (Batc  (None, 32, 32, 32)       128       
 hNormalization)                                                 
                                                                 
 re_lu_6 (ReLU)              (None, 32, 32, 32)        0         
                                                                 
 conv2d_7 (Conv2D)           (None, 32, 32, 32)        9216      
                                                                 
 batch_normalization_7 (Batc  (None, 32, 32, 32)       128       
 hNormalization)                                       

In [8]:
from tabnanny import verbose


model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)

history = model.fit(
    x_train, y_train, batch_size=64, epochs=100, validation_split=0.1, shuffle=True, verbose=2, callbacks=es
)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'test acc : {test_acc:.5f} | loss : {test_loss:.5f}')

Epoch 1/100
704/704 - 7s - loss: 1.3669 - accuracy: 0.5008 - val_loss: 1.7218 - val_accuracy: 0.4440 - 7s/epoch - 10ms/step
Epoch 2/100
704/704 - 6s - loss: 0.9686 - accuracy: 0.6569 - val_loss: 0.9282 - val_accuracy: 0.6708 - 6s/epoch - 9ms/step
Epoch 3/100
704/704 - 7s - loss: 0.8087 - accuracy: 0.7163 - val_loss: 1.4759 - val_accuracy: 0.5362 - 7s/epoch - 9ms/step
Epoch 4/100
704/704 - 7s - loss: 0.7065 - accuracy: 0.7533 - val_loss: 0.8795 - val_accuracy: 0.7022 - 7s/epoch - 10ms/step
Epoch 5/100
704/704 - 7s - loss: 0.6171 - accuracy: 0.7869 - val_loss: 0.9089 - val_accuracy: 0.6990 - 7s/epoch - 10ms/step
Epoch 6/100
704/704 - 7s - loss: 0.5519 - accuracy: 0.8093 - val_loss: 0.8778 - val_accuracy: 0.7070 - 7s/epoch - 10ms/step
Epoch 7/100
704/704 - 7s - loss: 0.4999 - accuracy: 0.8279 - val_loss: 0.8648 - val_accuracy: 0.7140 - 7s/epoch - 10ms/step
Epoch 8/100
704/704 - 7s - loss: 0.4579 - accuracy: 0.8438 - val_loss: 0.6419 - val_accuracy: 0.7896 - 7s/epoch - 10ms/step
Epoch 9/10